In [ ]:
# Install dependencies
!pip install -q torch torchvision timm Pillow scikit-learn matplotlib seaborn tqdm

import torch
import os

print(f'PyTorch version: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.2f} GB')

print('✓ Dependencies installed!')

## Step 1: Upload Data

**Option A: Use Kaggle Dataset (Recommended)**
1. Create a Kaggle Dataset with your data folders:
   - Upload `data/train/`, `data/val/`, `data/test/` folders
2. Add the dataset to this notebook: "Add data" → Your dataset
3. Update the path below

**Option B: Upload ZIP file**
- Upload `data.zip` and uncomment the unzip command below

In [ ]:
# Option A: If using Kaggle Dataset
# Replace 'yourusername/your-dataset-name' with your actual dataset path
DATA_DIR = '/kaggle/input/cervical-cancer-data'  # Change this to your dataset path

# Option B: If uploaded data.zip
# !unzip -q /kaggle/input/your-upload/data.zip -d /kaggle/working/
# DATA_DIR = '/kaggle/working/data'

# Verify data structure
print('Checking data structure...')
for split in ['train', 'val', 'test']:
    split_path = f'{DATA_DIR}/{split}'
    if os.path.exists(split_path):
        print(f'\n{split.upper()}:')
        for class_name in os.listdir(split_path):
            class_path = f'{split_path}/{class_name}'
            if os.path.isdir(class_path):
                count = len([f for f in os.listdir(class_path) if f.endswith(('.png', '.jpg', '.jpeg'))])
                print(f'  {class_name}: {count} images')
    else:
        print(f'⚠️ {split} folder not found at {split_path}')
        print(f'Please update DATA_DIR path above!')

In [ ]:
# Output files will be saved directly to /kaggle/working/
# (Kaggle's Output section works better with flat structure)

print('✓ Ready to save outputs to /kaggle/working/')

## Step 2: Model Architecture (Hybrid CNN + Vision Transformer)

In [ ]:
import torch
import torch.nn as nn
import timm

class HybridModel(nn.Module):
    def __init__(self, num_classes=5, use_pretrained=True):
        super().__init__()
        
        # CNN Branch: EfficientNet-B0
        self.cnn = timm.create_model('efficientnet_b0', pretrained=use_pretrained)
        cnn_features = self.cnn.classifier.in_features
        self.cnn.classifier = nn.Identity()
        
        # Vision Transformer Branch: ViT-Tiny
        self.vit = timm.create_model('vit_tiny_patch16_224', pretrained=use_pretrained)
        vit_features = self.vit.head.in_features
        self.vit.head = nn.Identity()
        
        # Fusion layers
        combined_features = cnn_features + vit_features
        self.fusion = nn.Sequential(
            nn.Linear(combined_features, 512),
            nn.ReLU(),
            nn.Dropout(0.5),
            nn.Linear(512, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )
    
    def forward(self, x):
        cnn_out = self.cnn(x)
        vit_out = self.vit(x)
        combined = torch.cat([cnn_out, vit_out], dim=1)
        return self.fusion(combined)

print('✓ Model architecture defined')

## Step 3: Focal Loss (For Minority Class Detection)

In [ ]:
import torch.nn.functional as F

class FocalLoss(nn.Module):
    def __init__(self, alpha=None, gamma=2.5, reduction='mean'):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction
    
    def forward(self, inputs, targets):
        ce_loss = F.cross_entropy(inputs, targets, reduction='none', weight=self.alpha)
        pt = torch.exp(-ce_loss)
        focal_loss = ((1 - pt) ** self.gamma) * ce_loss
        
        if self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()
        else:
            return focal_loss

print('✓ Focal Loss defined')

## Step 4: Data Loading with Augmentation

In [ ]:
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, WeightedRandomSampler
import numpy as np

# Training augmentation
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(20),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Validation/test transform (no augmentation)
val_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

# Load datasets
train_dataset = datasets.ImageFolder(f'{DATA_DIR}/train', transform=train_transform)
val_dataset = datasets.ImageFolder(f'{DATA_DIR}/val', transform=val_transform)

# Calculate class weights for Focal Loss
class_counts = np.bincount(train_dataset.targets)
class_weights = 1.0 / (class_counts ** 0.7)  # Aggressive weighting
class_weights = class_weights / class_weights.sum() * len(class_counts)
class_weights_tensor = torch.FloatTensor(class_weights)

# Sample weights for oversampling
sample_weights = np.array([class_weights[t] for t in train_dataset.targets])
sample_weights = sample_weights ** 0.8  # Additional emphasis
sampler = WeightedRandomSampler(sample_weights, len(sample_weights), replacement=True)

# Data loaders
train_loader = DataLoader(train_dataset, batch_size=64, sampler=sampler, num_workers=4)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False, num_workers=4)

print(f'\nDataset loaded:')
print(f'  Training samples: {len(train_dataset)}')
print(f'  Validation samples: {len(val_dataset)}')
print(f'  Classes: {train_dataset.classes}')
print(f'\nClass distribution:')
for i, class_name in enumerate(train_dataset.classes):
    print(f'  {class_name}: {class_counts[i]} samples, weight: {class_weights[i]:.3f}')

## Step 5: Training Setup

In [ ]:
# Initialize model
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = HybridModel(num_classes=len(train_dataset.classes)).to(device)

# Focal Loss with class weights
criterion = FocalLoss(alpha=class_weights_tensor.to(device), gamma=2.5)

# Optimizer with differential learning rates
optimizer = torch.optim.AdamW([
    {'params': model.cnn.parameters(), 'lr': 1e-4},
    {'params': model.vit.parameters(), 'lr': 1e-4},
    {'params': model.fusion.parameters(), 'lr': 1e-3}
], weight_decay=1e-4)

# Learning rate scheduler
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=100, eta_min=1e-6)

# Mixed precision training
scaler = torch.cuda.amp.GradScaler()

print('✓ Training setup complete')
print(f'Device: {device}')
print(f'Model parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M')

## Step 6: Training Loop

In [ ]:
from tqdm import tqdm
import time

def train_epoch(model, loader, criterion, optimizer, scaler, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(loader, desc='Training')
    for images, labels in pbar:
        images, labels = images.to(device), labels.to(device)
        
        optimizer.zero_grad()
        
        with torch.cuda.amp.autocast():
            outputs = model(images)
            loss = criterion(outputs, labels)
        
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        
        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()
        
        pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / total, 100. * correct / total

def validate(model, loader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []
    
    with torch.no_grad():
        pbar = tqdm(loader, desc='Validation')
        for images, labels in pbar:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = criterion(outputs, labels)
            
            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())
            
            pbar.set_postfix({'loss': f'{loss.item():.4f}', 'acc': f'{100.*correct/total:.2f}%'})
    
    return running_loss / total, 100. * correct / total, all_preds, all_labels

print('✓ Training functions defined')

In [ ]:
# Train the model
num_epochs = 100
best_val_acc = 0.0
patience = 15
patience_counter = 0

history = {
    'train_loss': [], 'train_acc': [],
    'val_loss': [], 'val_acc': []
}

print(f'Starting training for {num_epochs} epochs...')
print('='*80)

start_time = time.time()

for epoch in range(num_epochs):
    print(f'\nEpoch {epoch+1}/{num_epochs}')
    print('-' * 80)
    
    # Train
    train_loss, train_acc = train_epoch(model, train_loader, criterion, optimizer, scaler, device)
    
    # Validate
    val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)
    
    # Update scheduler
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Print summary
    print(f'\nTrain Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, '/kaggle/working/best_hybrid_model.pth')
        print(f'✓ Saved best model (Val Acc: {val_acc:.2f}%)')
        patience_counter = 0
    else:
        patience_counter += 1
    
    # Early stopping
    if patience_counter >= patience:
        print(f'\nEarly stopping at epoch {epoch+1}')
        break
    
    # Save checkpoint every 10 epochs
    if (epoch + 1) % 10 == 0:
        torch.save(model.state_dict(), f'/kaggle/working/checkpoint_epoch_{epoch+1}.pth')

total_time = time.time() - start_time
print(f'\n' + '='*80)
print(f'Training complete!')
print(f'Total time: {total_time/60:.2f} minutes')
print(f'Best validation accuracy: {best_val_acc:.2f}%')

## Step 7: Evaluate Best Model

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

# Load best model
checkpoint = torch.load('/kaggle/working/best_hybrid_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f'Loaded best model from epoch {checkpoint["epoch"]+1}')

# Evaluate
val_loss, val_acc, val_preds, val_labels = validate(model, val_loader, criterion, device)

print(f'\n' + '='*80)
print('CLASSIFICATION REPORT')
print('='*80)
print(classification_report(val_labels, val_preds, target_names=train_dataset.classes, digits=4))

# Confusion Matrix
cm = confusion_matrix(val_labels, val_preds)
plt.figure(figsize=(10, 8))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=train_dataset.classes, 
            yticklabels=train_dataset.classes)
plt.title('Confusion Matrix')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.savefig('/kaggle/working/confusion_matrix.png', dpi=300, bbox_inches='tight')
plt.show()

print('\n✓ Confusion matrix saved to /kaggle/working/confusion_matrix.png')

## Step 8: Plot Training History

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.savefig('/kaggle/working/training_history.png', dpi=300, bbox_inches='tight')
plt.show()

print('✓ Training history saved to /kaggle/working/training_history.png')

## Step 9: Download Trained Model

**The trained model is saved in the Output section:**
1. Click **"Data"** tab in the right sidebar
2. Navigate to **"Output"** section
3. Download these files:
   - `best_hybrid_model.pth` (best model)
   - `confusion_matrix.png`
   - `training_history.png`
   - `checkpoint_epoch_10.pth`, `checkpoint_epoch_20.pth`, etc. (if training went past 10 epochs)

In [ ]:
# List all saved files
print('Listing all output files...')
print('='*80)

# Check both locations (direct in /kaggle/working/ or in subdirectories)
!ls -lh /kaggle/working/*.pth /kaggle/working/*.png 2>/dev/null || echo "(checking subdirectories...)"
!ls -lh /kaggle/working/checkpoints/*.pth 2>/dev/null || true
!ls -lh /kaggle/working/outputs/*.png 2>/dev/null || true

# Create comprehensive zip with all outputs
print('\n' + '='*80)
print('Creating model_outputs.zip for download...')

# Zip everything - handles both flat and nested directory structures
!cd /kaggle/working && zip -r model_outputs.zip \
  *.pth *.png \
  checkpoints/*.pth \
  outputs/*.png \
  2>/dev/null || true

# Show final zip
print('\n')
!ls -lh /kaggle/working/model_outputs.zip

print('\n' + '='*80)
print('✓ All outputs packed in model_outputs.zip')
print('='*80)
print('\n📥 DOWNLOAD: Data → Output → model_outputs.zip')
print('\nContains:')
print('  - best_hybrid_model.pth (trained model)')
print('  - checkpoint_epoch_*.pth (checkpoints every 10 epochs)')
print('  - confusion_matrix.png (evaluation results)')
print('  - training_history.png (training curves)')
print('\n✓ Training complete!')